# مزوّد T5 مجاني لـ Ella2027

هذا الدفتر يشغّل نموذج `google/flan-t5-base` مجانًا داخل Google Colab، ويعرض واجهة API متوافقة مع `OpenAI Chat Completions` على المسار `/v1/chat/completions`.

> شغّل Colab على GPU من: Runtime → Change runtime type → T4 GPU إن كان متاحًا. النموذج عام ولا يحتاج Hugging Face token.

In [ ]:
!pip -q install -U transformers accelerate fastapi uvicorn nest-asyncio requests

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = 'google/flan-t5-base'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32

print('Device:', DEVICE)
print('Model:', MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
model.to(DEVICE)
model.eval()
print('T5 is ready')

In [ ]:
from typing import Any
from fastapi import FastAPI
from pydantic import BaseModel
import time, uuid, re, json

app = FastAPI(title='Ella2027 Free T5 Provider')

class ChatRequest(BaseModel):
    model: str | None = None
    messages: list[dict[str, Any]]
    temperature: float | None = 0.7
    max_tokens: int | None = 512

def make_prompt(messages: list[dict[str, Any]]) -> str:
    parts = []
    for message in messages:
        role = str(message.get('role', 'user')).upper()
        content = message.get('content', '')
        if isinstance(content, list):
            content = ' '.join(str(x.get('text', x)) if isinstance(x, dict) else str(x) for x in content)
        parts.append(f'{role}: {content}')
    return '\n'.join(parts) + '\nASSISTANT:'

def generate_text(prompt: str, max_tokens: int, temperature: float) -> str:
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(DEVICE)
    kwargs = {
        'max_new_tokens': max(16, min(int(max_tokens), 768)),
        'do_sample': temperature > 0,
        'num_beams': 2 if temperature <= 0 else 1,
        'temperature': max(0.1, min(float(temperature), 1.5)),
    }
    with torch.inference_mode():
        output = model.generate(**inputs, **kwargs)
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_ID, 'device': DEVICE}

@app.get('/v1/models')
def models():
    return {'object': 'list', 'data': [{'id': MODEL_ID, 'object': 'model', 'owned_by': 'google'}]}

@app.post('/v1/chat/completions')
def chat_completions(request: ChatRequest):
    prompt = make_prompt(request.messages)
    text = generate_text(prompt, request.max_tokens or 512, request.temperature or 0.7)
    now = int(time.time())
    return {
        'id': 'chatcmpl-' + uuid.uuid4().hex,
        'object': 'chat.completion',
        'created': now,
        'model': request.model or MODEL_ID,
        'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': text}, 'finish_reason': 'stop'}],
        'usage': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}
    }

print('API app is defined')

In [ ]:
# اختبار محلي داخل Colab قبل فتح النفق
from fastapi.testclient import TestClient
client = TestClient(app)
r = client.post('/v1/chat/completions', json={
    'model': MODEL_ID,
    'messages': [{'role': 'user', 'content': 'Reply with exactly: T5 is working'}],
    'max_tokens': 32,
    'temperature': 0
})
print(r.status_code)
print(r.json()['choices'][0]['message']['content'])

In [ ]:
# تشغيل API في الخلفية داخل Colab
import nest_asyncio, threading, uvicorn
nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
print('API started on http://127.0.0.1:8000')

In [ ]:
# إنشاء رابط عام مؤقت عبر Cloudflare Tunnel، بلا حساب Cloudflare
!wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null 2>&1 || true
import subprocess, time, re
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(30):
    line = tunnel.stdout.readline()
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)
print('Public URL:', public_url)
print('OpenAI-compatible base URL:', public_url + '/v1' if public_url else 'Tunnel not ready')

## ملاحظات الربط مع Ella2027

- رابط Colab يتغير عند إعادة تشغيل الجلسة، والجلسة المجانية قد تتوقف عند الخمول؛ لذلك هذا مناسب للتجربة وليس للاستضافة الدائمة.
- التطبيق الحالي لا يتيح إدخال `base URL` مخصص من الواجهة؛ ملف `providerRegistry.ts` يثبت Hugging Face على `https://router.huggingface.co/v1/chat/completions`.
- لإضافة هذا المزود داخل التطبيق، يجب إنشاء مزود باسم مثل `colab-t5` يرسل إلى `PUBLIC_URL/v1/chat/completions` ويستخدم النموذج `google/flan-t5-base`. لا تضع رابط النفق أو أي سر داخل GitHub العام.
- FLAN-T5-base نموذج صغير للتجربة وقد لا يلتزم دائمًا بإخراج JSON المعقد الذي يحتاجه مولّد الـ storyboard. إذا فشل JSON، استخدمه أولًا للاختبار أو بدّل إلى `google/flan-t5-large` عند توفر GPU وذاكرة كافية.
- لا تحفظ رابط النفق في المستودع؛ استخدم متغير بيئة في خادم Ella2027 أو أدخله يدويًا أثناء الاختبار.